In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
import os
from natsort import natsorted

import scanpy as sc
import seaborn as sns

from scroutines import basicu

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tools.sm_exceptions import ValueWarning
from tqdm import tqdm


import sys
sys.path.insert(0, '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/myvisctx/analysis_multiome/')
import lmm

In [2]:
%%time
outfigdir = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/'
f = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/superdupermegaRNA_hasraw_multiome_IT.h5ad'
adata_raw = sc.read(f)
adata_raw

CPU times: user 937 ms, sys: 9.43 s, total: 10.4 s
Wall time: 11.1 s


AnnData object with n_obs × n_vars = 89287 × 16567
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'time'
    var: 'feature_types'
    layers: 'norm'

In [3]:
f = '/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/data/v1_multiome/L4_labels_gao25_to_yoo25_knn.csv' 
df_lbl = pd.read_csv(f)
df_lbl

,label,conf
AATGTCATCGCACAAT-1-P21a-2023 Multiome-9-0,82_L4/5 IT CTX Glut_2,0.507621
GCTGATCCATTAGCGC-1-P21a-2023 Multiome-9-0,100_L4/5 IT CTX Glut_6,0.762219
CTCGCTCCAAAGCCTC-1-P21a-2023 Multiome-9-0,82_L4/5 IT CTX Glut_2,0.812493
AAACAGCCACCAAAGG-1-P21a-2023 Multiome-9-0,100_L4/5 IT CTX Glut_6,1.000000
AAACAGCCAGTAGGTG-1-P21a-2023 Multiome-9-0,100_L4/5 IT CTX Glut_6,0.936117
...,...,...
AGTACGCGTCATTAGG-1-P21DRa-2023 Multiome-10-0,100_L4/5 IT CTX Glut_6,0.592689
TATTTGGAGCAGGTTT-1-P21DRa-2023 Multiome-10-0,100_L4/5 IT CTX Glut_6,0.805783
CGGCTAATCAGTATTG-1-P21DRb-2023 Multiome-10-0,100_L4/5 IT CTX Glut_6,1.000000
TTAACTGAGTTACCGG-1-P21DRa-2023 Multiome-10-0,100_L4/5 IT CTX Glut_6,0.865567


In [4]:
adata = adata_raw[df_lbl.index].copy()
adata.obs = adata.obs.join(df_lbl)
adata.obs
adata

AnnData object with n_obs × n_vars = 5931 × 16567
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'time', 'label', 'conf'
    var: 'feature_types'
    layers: 'norm'

In [5]:
adata.X.data

array([23.,  2., 15., ...,  1.,  4.,  1.], dtype=float32)

In [6]:
adata.obs['Age'].unique()

['P21', 'P21DR']
Categories (2, object): ['P21', 'P21DR']

In [7]:
adata.obs['Sample'].unique()

['P21a', 'P21b', 'P21DRb', 'P21DRa']
Categories (4, object): ['P21DRa', 'P21DRb', 'P21a', 'P21b']

In [8]:
adata.obs['total_counts'].unique()

array([32499., 33681., 25103., ...,  8280.,  4731., 16848.], dtype=float32)

In [9]:
clusters = np.sort(adata.obs['label'].unique())
clusters

array(['100_L4/5 IT CTX Glut_6', '101_L4/5 IT CTX Glut_6',
       '68_L4/5 IT CTX Glut_1', '73_L4/5 IT CTX Glut_1',
       '82_L4/5 IT CTX Glut_2'], dtype=object)

In [10]:
import time

In [11]:
%%time

obs_fixed1 = 'Age'
obs_fixed2 = None # 'Light'
obs_random = 'Sample'

cluster_col = 'label'

offset = 1e-2
scale = 1e4

for cluster in clusters:
    tag = f"d260303_{cluster.replace('/', '').replace(' ', '_')}"
    output = os.path.join(outfigdir, f'NRDR_DEGs_LMM_yoo25_P21_{tag}.csv')

    adatasub = adata[adata.obs[cluster_col]==cluster]
    genes = adatasub.var.index.values 

    if obs_fixed2 is None:
        obs = adatasub.obs[[obs_fixed1, obs_random]].copy()
    else:
        obs = adatasub.obs[[obs_fixed1, obs_fixed2, obs_random]].copy()
    obs = obs.dropna()
    adatasub = adatasub[obs.index]

    # mat_raw = np.array(adatasub.X.todense())
    # mat_raw = np.array(adatasub.raw.X.todense())
    mat_raw = np.array(adatasub.X.todense())
    
    # ### test
    # adatasub = adatasub[:,:20]
    # genes = genes[:20]
    # mat_raw = mat_raw[:,:20]
    # ### test

    # mat (CP10k norm)
    # mat = mat_raw/adatasub.obs['n_counts'].values.reshape(-1,1)*scale
    mat = mat_raw/adatasub.obs['total_counts'].values.reshape(-1,1)*scale

    res = lmm.run_lmm(mat, genes, obs, obs_fixed1, obs_random, output_csv=output, offset=offset)
    print(output)

(4074, 16567) (4074, 2)
(4074, 16461) (4074, 2)
(4074, 8870) (4074, 2)
159 ['Zdbf2' 'Pth2r' 'Ptprn' 'Gm28294' 'Cops9' 'Bcl2' 'Cntnap5a' 'Dbi' 'Btg2'
 'Glul' 'Atp1a2' 'Camk1g' 'Pfkfb3' 'Arl5b' 'Ptgds' 'Pbx3' 'Lypd6' 'Rnd3'
 'Rprm' 'Nr4a2' 'Tanc1' 'Bdnf' 'Cst3' 'Cbln4' 'Rps21' 'Rpl39' 'Xist'
 'Plp1' 'Car2' 'Skil' 'Tiparp' 'Bcan' 'S100a1' 'S100a13' 'S100a16' 'Gstm5'
 'Gstm1' 'Ddit4l' 'Gm12371' 'Tgfbr1' 'Nr4a3' 'Ak4' 'Stk40' 'Tnfrsf25'
 'Fosl2' 'Galnt9' 'Mn1' 'Sgsm1' 'Tsc22d4' 'Nptx2' 'Pon2' 'Ptprz1' 'Pde1c'
 'Rpl32' 'B4galnt3' 'P3h3' 'Lmo3' 'Prkd2' 'Fosb' 'Apoe' 'Nfkbid' 'Mag'
 'Akap13' 'Mapk3' 'Plagl1' 'Egr2' 'Fam13c' 'S100b' 'Midn' 'Gadd45b' 'Nab2'
 'Cd63' 'Irs2' 'Prag1' 'Vegfc' 'Scrg1' 'Galnt7' 'Tpm4' 'Junb' 'Mt3' 'Mt2'
 'Mt1' 'Cdyl2' 'Cbfa2t3' 'Fam107a' 'Anxa11' 'Ndrg2' 'Clu' 'Egr3' 'Cldn10'
 'Sik2' 'Gm17231' 'Arid3b' 'Tle3' 'Rplp1' 'Smad3' 'Trim71' 'Mobp' 'Spred2'
 'Gm2a' 'Sparc' 'Kdm6b' '1700016P03Rik' 'Aldoc' 'Fmnl1' 'Map3k14'
 'Hist1h2bc' 'Gfod1' 'Dok3' 'Gm47423' 'Pcsk1' 'Homer1' 

100% 8870/8870 [15:22<00:00,  9.62it/s]


66 ['Zdbf2' 'Gm28294' 'Bcl2' 'Btg2' 'Camk1g' 'Pfkfb3' 'Pbx3' 'Rnd3' 'Nr4a2'
 'Tanc1' 'Xist' 'Skil' 'Tiparp' 'Bcan' 'Tnfrsf25' 'Sgsm1' 'Tsc22d4'
 'B4galnt3' 'P3h3' 'Lmo3' 'Prkd2' 'Fosb' 'Nfkbid' 'Akap13' 'Mapk3' 'Egr2'
 'Fam13c' 'Midn' 'Nab2' 'Prag1' 'Galnt7' 'Cdyl2' 'Cbfa2t3' 'Anxa11' 'Egr3'
 'Cldn10' 'Sik2' 'Arid3b' 'Smad3' 'Spred2' 'Gm2a' '1700016P03Rik' 'Fmnl1'
 'Map3k14' 'Gfod1' 'Dok3' 'Pcsk1' 'Homer1' 'Frmd6' 'Inf2' 'Klf10' 'Zhx2'
 'Trib1' 'Myh9' 'Ccdc134' 'Phf21b' 'Grasp' 'Etv5' 'Synj2' 'Sox8' 'Grm4'
 'Sik1' 'Plekhh2' 'Eif2s3y' 'Uty' 'Ddx3y']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_100_L45_IT_CTX_Glut_6.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_100_L45_IT_CTX_Glut_6.csv
(382, 16567) (382, 2)
(382, 15237) (382, 2)
(382, 9176) (382, 2)
512 ['Rgs20' 'Mcmdc2' 'Snhg6' 'Cnnm4' '9330175M20Rik' 'Stat4' '9130024F11Rik'
 'Gm15834' 'Zdbf2' 'Ikzf2' 'Tmem169' 'Xrcc5' 'Tmem1

100% 9176/9176 [07:39<00:00, 19.95it/s]


84 ['Rgs20' 'Mcmdc2' 'Ikzf2' 'Btg2' 'Rnd3' 'Mettl8' 'Itga4' 'Xist' 'Ddah1'
 'Gem' '1700123M08Rik' 'B4galt1' 'Tgfbr1' 'Epha10' 'Rspo1' 'Serinc2' 'Hgf'
 'Drc1' 'Fosl2' 'Galnt9' 'Sgsm1' 'Rph3a' 'Vgf' 'Il17ra' 'Stk38l' 'Gm21814'
 'Fosb' 'Pvr' 'Sbk1' 'Lama2' 'Fam13c' 'Phlda1' 'Irs2' 'Ank1' 'Prag1'
 'Msmo1' 'Junb' 'Cdyl2' 'Slc25a37' 'Egr3' 'Aasdhppt' 'Pde4a' 'Ppp2r1b'
 'Sik2' 'Dnaja4' 'Gm17231' 'Arid3b' 'Pgm3' 'Spred2' 'Gm39822'
 '1700016P03Rik' 'Dusp14' 'Arhgap23' 'Stat3' 'Tmem170b' 'Gfod1' 'Pcsk1'
 'Homer1' 'Hmgcr' 'Gcnt4' 'Mccc2' 'Rock2' 'Gpr68' 'Inf2' 'Zhx2' 'Arc'
 'Eif3l' 'Pdgfb' 'Ccdc134' 'Scube1' 'Lrrk2' 'Grasp' 'Nr4a1' 'Etv5'
 'Plcxd2' 'Dusp1' 'Galnt14' 'Plekhh2' 'Eif2s3y' 'Uty' 'Mapk4' 'Npas4'
 'Smndc1' 'Dusp5']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_101_L45_IT_CTX_Glut_6.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_101_L45_IT_CTX_Glut_6.csv
(353, 16567) (353, 2)
(3

...

82 ['Pdcl3' 'Coq10b' 'Zdbf2' 'Resp18' 'Btg2' 'Tnfaip6' 'Nr4a2' 'Gm13889'
 'Bdnf' 'Sulf2' 'Med14' 'Gm14827' 'Xist' 'Tiparp' 'Lmna' 'Kcnn3' 'Dennd4b'
 'Cd101' 'Dnajb5' 'Nr4a3' 'Scp2' 'Rnf19b' 'Rheb' 'Fosl2' 'Acox3' 'Mn1'
 'Sgsm1' 'Rph3a' 'Vgf' 'Gm15411' 'Gpr19' 'Fosb' 'Numbl' 'Stx4a' 'Egr2'
 'Phlda1' 'Irs2' 'Prag1' 'Mast3' 'Pard3' 'Slc25a37' 'Egr3' 'Kctd4' 'Pde4a'
 'Zbtb16' 'Sik2' 'Gm17231' 'Tle3' 'Lyzl4' 'Spred2' 'Ccnjl' 'Camkk1'
 '1700016P03Rik' 'Mettl23' 'Hivep1' 'Gfod1' 'Gm47423' 'Pcsk1' 'Homer1'
 'Rock2' 'Frmd6' 'Fam71d' 'Ahsa1' 'Traf3' 'Trib1' 'Phf21b' 'Grasp' 'Nr4a1'
 'Etv5' 'Plcxd2' 'Synj2' 'Tedc2' 'Sik1' 'Mrps10' 'Plekhh2' 'Eif2s3y' 'Uty'
 'Ddx3y' 'Adamts19' 'Npas4' 'Nolc1' 'Gfra1']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_68_L45_IT_CTX_Glut_1.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_68_L45_IT_CTX_Glut_1.csv
(974, 16567) (974, 2)
(974, 16152) (974, 2)
(974, 86

100% 8688/8688 [18:02<00:00,  8.03it/s]


30 ['Rassf5' 'Nr4a2' 'Rims4' 'Xist' 'Tiparp' 'Gm17501' 'Fosl2' 'Mn1' 'Ttll3'
 'Akap13' 'Prag1' 'Anxa11' 'Arhgef3' 'Egr3' 'Sik2' 'Lyzl4' '1700016P03Rik'
 'Homer1' 'Scube1' 'Phf21b' 'Grasp' 'Pfdn5' 'Plcxd2' 'Sik1' 'Plekhh2'
 'Kdm5d' 'Eif2s3y' 'Uty' 'Ddx3y' 'Sorcs3']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_73_L45_IT_CTX_Glut_1.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_73_L45_IT_CTX_Glut_1.csv
(148, 16567) (148, 2)
(148, 14525) (148, 2)
(148, 9903) (148, 2)
1399 ['Snhg6' 'Prex2' 'Rdh10' ... 'mt-Nd3' 'mt-Nd4' 'mt-Nd6']


100% 9903/9903 [09:59<00:00, 16.53it/s]


62 ['Nrp2' 'Cdh20' 'Btg2' 'Glul' 'Gm38251' 'Ppp2r5a' 'Dnajc1' 'Dnlz'
 'Gm14471' 'Spred1' 'Egfem1' 'Gm12743' 'Wasf2' 'C1qtnf12' 'Uvssa' 'Rad9b'
 'Selenow' 'Apoe' 'Timm50' 'Mrps12' 'Zfp940' 'Pcf11' '1700012D14Rik'
 'Gpr26' 'Egr2' 'Sgta' 'Stac3' 'Prag1' 'Psmb10' 'Cog8' 'Zfp612' 'Cntnap4'
 'Cbfa2t3' 'Synpr' 'Egr3' 'Elof1' 'Igsf9b' 'Nprl2' 'Rel' '1700016P03Rik'
 'Nt5c' 'Baiap2' 'Gfod1' 'Pcsk1' 'Dus4l' 'Kcnv1' 'Arc' 'Ccdc134' 'Scube1'
 'Marf1' 'Pak2' 'Btg3' 'Grik1' 'Dusp1' 'Sik1' 'H2-Ke6' 'Ubxn6' 'Plekhh2'
 'Uty' 'Vegfb' 'Syt7' 'mt-Nd6']
save to csv: /u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_82_L45_IT_CTX_Glut_2.csv
/u/home/f/f7xiesnm/project-zipursky/v1-bb/v1/results_v1astro/NRDR_DEGs_LMM_yoo25_P21_d260303_82_L45_IT_CTX_Glut_2.csv
CPU times: user 59min 38s, sys: 30.5 s, total: 1h 9s
Wall time: 1h 11s


In [12]:
adata

AnnData object with n_obs × n_vars = 5931 × 16567
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'time', 'label', 'conf'
    var: 'feature_types'
    layers: 'norm'